# 11 · EU LNG Storage Analysis

Analysis of EU LNG terminal fill rates, send-out rates and their relationship
with gas storage levels and TTF prices.

**Data sources:**
- ALSI+ API (https://alsi.gie.eu) — LNG storage
- EU gas storage: `data/processed/eu_aggregate_full.parquet`
- TTF forward curve: `data/raw/ttf_curve.csv`

Paste your ALSI API key in cell 0 below, then run all cells.

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import date
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c)); os.chdir(_c)
        print(f"\u2705 Root: {_c}"); break

# ═════════════════════════════════════════════════════════════════
# CONFIGURATION  — paste your key here
# ═════════════════════════════════════════════════════════════════
API_KEY       = "paste_key_here"
ANALYSIS_DATE = None          # None = last data point
START_DATE    = "2020-01-01"
# ═════════════════════════════════════════════════════════════════

from src.agsi_client.alsi_client import ALSIClient

client = ALSIClient(api_key=API_KEY, ssl_verify=False)

storage_path = Path("data/processed/eu_aggregate_full.parquet")
ttf_path     = Path("data/raw/ttf_curve.csv")

storage_df = pd.read_parquet(storage_path) if storage_path.exists() else pd.DataFrame()
ttf_df     = pd.read_csv(ttf_path, index_col=0, parse_dates=True) if ttf_path.exists() else pd.DataFrame()

if not storage_df.empty:
    storage_df = storage_df[storage_df.index >= START_DATE].sort_index()
if not ttf_df.empty:
    ttf_df = ttf_df[ttf_df.index >= START_DATE].sort_index()


# Strip timezone info — TTF CSV is tz-aware (UTC), parquet is tz-naive
for _df in [ttf_df, storage_df]:
    if not _df.empty and hasattr(_df.index, "tz") and _df.index.tz is not None:
        _df.index = _df.index.tz_localize(None)
print("Setup complete. Run cell 1 to fetch LNG data.")

## 1 · Fetch EU LNG Aggregate & Save

In [ ]:
lng_df = client.get_eu_aggregate(start=START_DATE)

if lng_df.empty:
    print("\u26a0\ufe0f  No LNG data returned. Check API key and network.")
else:
    out_path = Path("data/processed/eu_lng_full.parquet")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    lng_df.to_parquet(out_path)
    print(f"\u2705 Saved {len(lng_df)} rows \u2192 {out_path}")
    print(lng_df.tail(5)[["inventory", "sendOut", "dtmi", "full"]])


# Strip timezone from LNG index if tz-aware
if not lng_df.empty and hasattr(lng_df.index, "tz") and lng_df.index.tz is not None:
    lng_df.index = lng_df.index.tz_localize(None)
analysis_date = (
    ANALYSIS_DATE if ANALYSIS_DATE
    else (lng_df.index[-1].date() if not lng_df.empty else date.today())
)
print(f"\nAnalysis date: {analysis_date}")


## 2 · LNG Fill Rate vs TTF Price

In [ ]:
if lng_df.empty or ttf_df.empty:
    print("\u26a0\ufe0f  Need both LNG and TTF data.")
else:
    merged = lng_df[["full"]].rename(columns={"full": "lng_full"}).join(
        ttf_df[["M1"]].rename(columns={"M1": "ttf_m1"}), how="inner"
    ).dropna()

    corr = merged["lng_full"].corr(merged["ttf_m1"])

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["TTF M1 Price (\u20ac/MWh)", "EU LNG Fill Rate (%)"],
                        vertical_spacing=0.08, row_heights=[0.55, 0.45])

    fig.add_trace(go.Scatter(x=merged.index, y=merged["ttf_m1"],
                             name="TTF M1", line=dict(color="#6366f1", width=1.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=merged.index, y=merged["lng_full"],
                             name="LNG Fill %", fill="tozeroy",
                             line=dict(color="#0ea5e9", width=1.5),
                             fillcolor="rgba(14,165,233,0.15)"), row=2, col=1)

    fig.update_layout(height=500, template="plotly_white",
                      title=f"LNG Fill Rate vs TTF M1 (Pearson r = {corr:.2f})",
                      legend=dict(orientation="h", y=-0.12))
    fig.show()

    # Scatter
    fig2 = go.Figure(go.Scatter(
        x=merged["lng_full"], y=merged["ttf_m1"],
        mode="markers",
        marker=dict(color="#0ea5e9", size=4, opacity=0.5)
    ))
    fig2.update_layout(template="plotly_white", height=400,
                       title="LNG Fill Rate vs TTF M1 \u2014 Scatter",
                       xaxis_title="LNG Fill Rate (%)", yaxis_title="TTF M1 (\u20ac/MWh)")
    fig2.show()

## 3 · LNG Send-Out Rate Over Time

In [ ]:
if lng_df.empty:
    print("\u26a0\ufe0f  No LNG data.")
elif "sendOut" not in lng_df.columns:
    print("\u26a0\ufe0f  sendOut column not available.")
else:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=lng_df.index, y=lng_df["sendOut"],
                             name="EU LNG Send-Out", fill="tozeroy",
                             line=dict(color="#0ea5e9", width=1.5),
                             fillcolor="rgba(14,165,233,0.12)"))

    # Rolling average
    rolling = lng_df["sendOut"].rolling(30).mean()
    fig.add_trace(go.Scatter(x=lng_df.index, y=rolling,
                             name="30d avg", line=dict(color="#1e40af", width=2, dash="dash")))

    fig.update_layout(template="plotly_white", height=420,
                      title="EU LNG Send-Out Rate (GWh/day)",
                      yaxis_title="GWh/day", xaxis_title="Date",
                      legend=dict(orientation="h", y=-0.12))
    fig.show()
    print(f"Latest send-out: {lng_df['sendOut'].dropna().iloc[-1]:,.0f} GWh/day")
    print(f"1-year avg: {lng_df['sendOut'].last('365D').mean():,.0f} GWh/day")

## 4 · LNG Fill Rate vs Gas Storage Fill Rate (Dual Chart)

In [ ]:
if lng_df.empty or storage_df.empty:
    print("\u26a0\ufe0f  Need both LNG and gas storage data.")
else:
    merged2 = lng_df[["full"]].rename(columns={"full": "lng_full"}).join(
        storage_df[["full"]].rename(columns={"full": "gas_full"}), how="inner"
    ).dropna()

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=merged2.index, y=merged2["gas_full"],
                             name="Gas Storage Fill %",
                             line=dict(color="#22c55e", width=2)))
    fig.add_trace(go.Scatter(x=merged2.index, y=merged2["lng_full"],
                             name="LNG Fill %",
                             line=dict(color="#0ea5e9", width=2)))

    fig.update_layout(template="plotly_white", height=430,
                      title="EU Gas Storage Fill vs EU LNG Fill Rate (%)",
                      yaxis_title="Fill Rate (%)", xaxis_title="Date",
                      legend=dict(orientation="h", y=-0.12))
    fig.show()

    corr2 = merged2["gas_full"].corr(merged2["lng_full"])
    print(f"Correlation (gas fill vs LNG fill): {corr2:.3f}")

## 5 · Combined Energy Buffer (Gas + LNG)

In [ ]:
if lng_df.empty or storage_df.empty:
    print("\u26a0\ufe0f  Need both LNG and gas storage data.")
else:
    gas = storage_df[["gasInStorage"]].rename(columns={"gasInStorage": "gas_TWh"})
    lng = lng_df[["inventory"]].rename(columns={"inventory": "lng_TWh"})

    combined = gas.join(lng, how="inner").dropna()
    combined["total_TWh"] = combined["gas_TWh"] + combined["lng_TWh"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=combined.index, y=combined["total_TWh"],
                             name="Total (Gas + LNG)", fill="tozeroy",
                             line=dict(color="#7c3aed", width=2),
                             fillcolor="rgba(124,58,237,0.10)"))
    fig.add_trace(go.Scatter(x=combined.index, y=combined["gas_TWh"],
                             name="Gas Storage", line=dict(color="#22c55e", width=1.5)))
    fig.add_trace(go.Scatter(x=combined.index, y=combined["lng_TWh"],
                             name="LNG Storage", line=dict(color="#0ea5e9", width=1.5)))

    fig.update_layout(template="plotly_white", height=460,
                      title="Combined EU Energy Buffer \u2014 Gas Storage + LNG (TWh)",
                      yaxis_title="TWh", xaxis_title="Date",
                      legend=dict(orientation="h", y=-0.12))
    fig.show()

    latest = combined.iloc[-1]
    print(f"\nLatest ({combined.index[-1].date()}):")
    print(f"  Gas storage : {latest['gas_TWh']:,.0f} TWh")
    print(f"  LNG storage : {latest['lng_TWh']:,.0f} TWh")
    print(f"  Combined    : {latest['total_TWh']:,.0f} TWh")
